# 第 10 章 · 归约：一百万个数求和，怎么让一千个线程分着加

本章回答：求和、求最大值这类"很多个数变成一个数"的运算，在 GPU 上怎么并行？block 内的树形归约怎么写？为什么"前一半加后一半"比"相邻两两相加"好？block 之间的部分和怎么合起来？

配套讲义：仓库里 `cuda-programming/notes/lessons/10-reduction.md`（讲解、动画、练习都在讲义里）。对应原文：Oxford lec4 第 7–18 页（`materials/text/oxford-lec4.txt` 第 162–395 行，局部归约代码在第 15 页 / 第 336–352 行）；PMPP 第 5 版 ch.10 Reduction；本仓库 `solutions/cubecl-lab/src/lib.rs` M1 的 TODO 注释；Mark Harris「Optimizing Parallel Reduction in CUDA」（〔旧知〕在线）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 浮点不结合 | $(a+b)+c \ne a+(b+c)$ | `(0.1 + 0.2) + 0.3` vs `0.1 + (0.2 + 0.3)` | 0.6000000000000001 vs 0.6 |
| 第二步 前一半加后一半 | `temp[t] += temp[t + d]` | `tree_halves([3, 1, 4, 1, 5, 9, 2, 6])` | 8 10 6 7 → 14 17 → 31，3 轮 |
| 第三步 两种写法干活的线程 | 连续 vs 隔一个 | `active_threads_halves(8)` / `active_threads_interleaved(8)` | [[0,1,2,3],[0,1],[0]] vs [[0,2,4,6],[0,4],[0]] |
| 第三步 bank conflict | 跨步 4 | `conflict_ways(np.arange(32) * 4)`（第 7 章的函数） | 4 路 |
| 第五步 grid-stride | `i += gridDim.x * blockDim.x` | `grid_stride_partials(data, 2, 4)` | 线程 0 读 0、8、16；总和 210 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `active_threads_halves`**（来自 `ch10_reduction.py`）：前一半加后一半：每一轮干活的线程号。

In [ ]:
def active_threads_halves(n: int) -> list[list[int]]:
    """前一半加后一半：每一轮干活的线程号。"""
    out, d = [], n // 2
    while d > 0:
        out.append(list(range(d)))
        d //= 2
    return out

**函数 `active_threads_interleaved`**（来自 `ch10_reduction.py`）：相邻两两相加：每一轮干活的线程号。

In [ ]:
def active_threads_interleaved(n: int) -> list[list[int]]:
    """相邻两两相加：每一轮干活的线程号。"""
    out, s = [], 1
    while s < n:
        out.append([t for t in range(n) if t % (2 * s) == 0])
        s *= 2
    return out

**常量 `BANKS`**（来自 `ch07_bank_conflict.py`）

In [ ]:
BANKS = 32

**函数 `bank`**（来自 `ch07_bank_conflict.py`）：字地址（以 4 字节为单位）→ bank 号。

In [ ]:
def bank(word_addr):
    """字地址（以 4 字节为单位）→ bank 号。"""
    return np.asarray(word_addr) % BANKS

**函数 `conflict_ways`**（来自 `ch07_bank_conflict.py`）：几路冲突 = 同一个 bank 上不同地址最多有几个（同一地址算广播，只算一个）。

In [ ]:
def conflict_ways(word_addrs) -> int:
    """几路冲突 = 同一个 bank 上不同地址最多有几个（同一地址算广播，只算一个）。"""
    distinct = np.unique(np.asarray(word_addrs))
    return int(np.bincount(bank(distinct), minlength=BANKS).max())

**函数 `grid_stride_partials`**（来自 `ch10_reduction.py`）：每个线程从 i = 全局编号 开始，每次跳过整个 grid 的线程数，在"寄存器"里累加。返回每个线程的 acc。

In [ ]:
def grid_stride_partials(data: np.ndarray, grid_dim: int, block_dim: int) -> np.ndarray:
    """每个线程从 i = 全局编号 开始，每次跳过整个 grid 的线程数，在"寄存器"里累加。返回每个线程的 acc。"""
    total = grid_dim * block_dim
    acc = np.zeros(total, dtype=data.dtype)
    for tid in range(total):
        acc[tid] = data[tid::total].sum()   # i = tid, tid + total, tid + 2·total, …
    return acc

**函数 `tree_halves`**（来自 `ch10_reduction.py`）：前一半加后一半：for d = n/2 … 1：前 d 个线程 temp[t] += temp[t + d]。返回每一轮之后的 temp。

In [ ]:
def tree_halves(v: list[float]) -> list[list[float]]:
    """前一半加后一半：for d = n/2 … 1：前 d 个线程 temp[t] += temp[t + d]。返回每一轮之后的 temp。"""
    temp = list(v)
    rows = [temp[:]]
    d = len(temp) // 2
    while d > 0:
        for t in range(d):           # 同一轮里这些线程是同时做的（之前有一道 __syncthreads）
            temp[t] += temp[t + d]
        rows.append(temp[:])
        d //= 2
    return rows

**函数 `tree_interleaved`**（来自 `ch10_reduction.py`）：相邻两两相加：for s = 1, 2, 4…：tid % (2s) == 0 的线程 temp[t] += temp[t + s]。

In [ ]:
def tree_interleaved(v: list[float]) -> list[list[float]]:
    """相邻两两相加：for s = 1, 2, 4…：tid % (2s) == 0 的线程 temp[t] += temp[t + s]。"""
    temp = list(v)
    rows = [temp[:]]
    s = 1
    while s < len(temp):
        for t in range(0, len(temp), 2 * s):
            temp[t] += temp[t + s]
        rows.append(temp[:])
        s *= 2
    return rows

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
v = [3, 1, 4, 1, 5, 9, 2, 6]

In [ ]:
print("\n第一步 浮点加法不严格结合")
print(f"  (0.1 + 0.2) + 0.3 = {(0.1 + 0.2) + 0.3!r}，0.1 + (0.2 + 0.3) = {0.1 + (0.2 + 0.3)!r}")
x = np.random.default_rng(0).random(1_000_000).astype(np.float32)
seq = np.cumsum(x, dtype=np.float32)[-1]
tree = x.sum(dtype=np.float32)
print(f"  100 万个 float32：顺序加 = {seq:.2f}，分组加 = {tree:.2f}，差 {abs(seq - tree):.2f}")

In [ ]:
print("\n第二步 前一半加后一半，blockDim = 8")
for k, row in enumerate(tree_halves(v)):
    print(f"  第 {k} 轮后：{row}")
print("  轮数 = log2(8) =", len(tree_halves(v)) - 1, "；加法次数 =", sum(len(a) for a in active_threads_halves(8)))

In [ ]:
print("\n第三步 相邻两两相加：结果一样，干活的线程不一样")
print("  结果：", tree_interleaved(v)[-1][0])
print("  前一半加后一半，每轮干活的线程：", active_threads_halves(8))
print("  相邻两两相加，每轮干活的线程：  ", active_threads_interleaved(8))
print("  相邻两两相加第 2 轮：线程 t 访问 temp[4k]，跨步 4 →", conflict_ways(np.arange(32) * 4), "路冲突")
print("  前一半加后一半：连续线程读连续地址 →", conflict_ways(np.arange(32)), "路")

In [ ]:
print("\n第五步 grid-stride loop：2 个 block × 4 个线程，处理 20 个数")
data = np.arange(1, 21, dtype=np.float64)
acc = grid_stride_partials(data, 2, 4)
print("  线程 0 依次读的下标：", list(range(0, 20, 8)))
print("  8 个线程各自的 acc：", acc)
print("  再做树形归约：", tree_halves(list(acc))[-1][0], "= sum(1..20) =", data.sum())

## 3. 代码里才看得到的

1. **Python 的 for 循环是一个一个做的**：同一轮里"前 d 个线程同时做"在这里写成循环；能这样写是因为同一轮里每个线程读写的位置互不重叠（读 `t + d`、写 `t`）。
2. **`data[tid::total]` 就是 grid-stride 的下标序列**：从自己的全局编号开始，每次加 grid 的总线程数。
3. **100 万个 float32 两种加法顺序结果不同**：`np.cumsum` 是一个一个顺序加，`np.sum` 内部分组两两加，最后几位不一样——GPU 和 CPU 对拍要用容差（第 27 章）。

## 4. 自己改一改

- 把 `tree_halves` 的输入换成 16 个数，看要几轮。
- 把 `grid_stride_partials` 的 grid / block 换成 (1, 4)、(5, 4)，确认总和不变。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 能结合就能并行，浮点不严格结合，对拍要容差；
> 淘汰赛：N 个数 log N 轮；
> 前一半加后一半：不分岔、不冲突；相邻两两加：又分岔又冲突；
> block 之间不能等：要么两个 kernel，要么原子加；
> 先在寄存器里多加几个（grid-stride），再进 shared 打淘汰赛。